### Dataset

In [9]:
import os

def count_trains(split_dir):
    labels_dir = os.path.join(split_dir, "labels_og")

    images_with_trains = 0
    images_without_trains = 0
    images_with_multiple_trains=0
    total_boxes = 0

    for label_file in os.listdir(labels_dir):
        if not label_file.endswith(".txt"):
            continue

        label_path = os.path.join(labels_dir, label_file)

        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        if len(lines) > 0:
            images_with_trains += 1
            total_boxes += len(lines)
            if len(lines) > 1:
                images_with_multiple_trains+=1
        else:
            images_without_trains += 1

    total_images = images_with_trains + images_without_trains

    return {
        "total_images": total_images,
        "images_with_trains": images_with_trains,
        "images_with_multiple_trains": images_with_multiple_trains,
        "images_without_trains": images_without_trains,
        "total_train_instances": total_boxes
    }


# Root directory of your dataset
dataset_root = "/data22/datasets/zollner_dataset/paper_data/"

for split in ["train", "val", "test"]:
    split_path = os.path.join(dataset_root, split)
    stats = count_trains(split_path)

    print(f"\n📁 {split.upper()} SET")
    for k, v in stats.items():
        print(f"{k}: {v}")


📁 TRAIN SET
total_images: 5600
images_with_trains: 4519
images_with_multiple_trains: 18
images_without_trains: 1081
total_train_instances: 4537

📁 VAL SET
total_images: 700
images_with_trains: 569
images_with_multiple_trains: 2
images_without_trains: 131
total_train_instances: 571

📁 TEST SET
total_images: 700
images_with_trains: 563
images_with_multiple_trains: 4
images_without_trains: 137
total_train_instances: 567


In [10]:
# import cv2
# import os
# import numpy as np
# from collections import Counter

# def classify_time_of_day(image_path):
#     img = cv2.imread(image_path)
#     if img is None:
#         return "unknown"

#     # Resize for speed
#     img = cv2.resize(img, (256, 256))

#     # ----- Brightness (HSV) -----
#     hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
#     brightness = np.mean(hsv[:, :, 2])

#     # ----- Color warmth (BGR) -----
#     b, g, r = cv2.split(img)
#     warmth = np.mean(r) - np.mean(b)

#     # ----- Heuristic thresholds -----
#     if brightness < 50:
#         return "night"
#     elif brightness < 120:
#         return "evening"
#     else:
#         if warmth > 15:
#             return "evening"
#         else:
#             return "day"


# def analyze_split(split_dir):
#     images_dir = os.path.join(split_dir, "images")
#     counts = Counter()

#     for img_file in os.listdir(images_dir):
#         if not img_file.lower().endswith((".jpg", ".png", ".jpeg")):
#             continue

#         img_path = os.path.join(images_dir, img_file)
#         tod = classify_time_of_day(img_path)
#         counts[tod] += 1

#     return counts


# dataset_root = "/data22/datasets/zollner_dataset/paper_data/"

# for split in ["train", "val", "test"]:
#     split_path = os.path.join(dataset_root, split)
#     stats = analyze_split(split_path)

#     print(f"\n📁 {split.upper()} SET")
#     for k, v in stats.items():
#         print(f"{k}: {v}")



📁 TRAIN SET
evening: 1562
day: 2157
night: 1881

📁 VAL SET
night: 211
day: 292
evening: 197

📁 TEST SET
day: 274
evening: 182
night: 244


In [5]:
import cv2
import os
import random
import shutil
import numpy as np
from collections import defaultdict
from tqdm import tqdm

def classify_day_night_old(image_path, threshold=80):
    img = cv2.imread(image_path)
    if img is None:
        return "unknown"

    img = cv2.resize(img, (256, 256))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    brightness = np.mean(hsv[:, :, 2])

    return "night" if brightness < threshold else "day"


def classify_day_night(image_path, threshold=60):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")

    height, width = img.shape[:2]

    # Define ROI: top-right corner (adjust size if needed)
    roi = img[0:int(0.25*height), int(0.75*width):width]  # top 25% height, right 25% width

    # Convert ROI to grayscale
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # Compute average brightness
    avg_brightness = np.mean(gray)

    # Decide day/night
    if avg_brightness > threshold:
        return "Day"
    else:
        return "Night"


def classify_day_night_roi(image_path, threshold=60):
    img = cv2.imread(image_path)
    if img is None:
        return "unknown"

    h, w, _ = img.shape

    # ---- Crop top-right corner (sky region) ----
    roi = img[0:int(0.25 * h), int(0.65 * w):w]

    # Convert to HSV
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    brightness = np.mean(hsv[:, :, 2])

    return "night" if brightness < threshold else "day"


dataset_root = "/data22/datasets/zollner_dataset/paper_data/"
output_root = "manual_check_day_night"
os.makedirs(output_root, exist_ok=True)

images_by_class = defaultdict(list)

# Collect predictions
for split in ["train", "val", "test"]:
    images_dir = os.path.join(dataset_root, split, "images")
    for img_file in tqdm(os.listdir(images_dir)):
        if img_file.lower().endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(images_dir, img_file)
            pred = classify_day_night_roi(img_path)
            if pred in ["day", "night"]:
                images_by_class[pred].append(img_path)

# Sample & copy
SAMPLES_PER_CLASS = 50

for cls in ["day", "night"]:
    os.makedirs(os.path.join(dataset_root,output_root, cls), exist_ok=True)

    for img_path in tqdm(images_by_class[cls]):
        #make label file of type txt and add day or night to it accordingly
        label_path = os.path.join(dataset_root,output_root, cls, os.path.basename(img_path).replace(".jpg", ".txt").replace(".png", ".txt"))
        # print(f"Creating label file: {label_path}")
        with open(label_path, "w") as f:
            f.write(cls)        

100%|██████████| 3688/3688 [00:00<00:00, 22502.95it/s]


In [41]:
print(f"Day time images: {len(images_by_class['day'])}- {100*len(images_by_class['day'])/7000} \nNight time images: {len(images_by_class['night'])} - {100*len(images_by_class['night'])/7000}")

Day time images: 3312- 47.31428571428572 
Night time images: 3688 - 52.68571428571428
